# Domain-level Z3 Optimize objective rules

**Status:** Draft specification for user review  
**Date:** 2026-08-24  
**Design epic:** `bd-27mx`  
**Plan ID:** `c41985a8-7494-44d8-9581-8a0af61d24b1`

This notebook is the authoritative design for Wave 1 of domain-level optimization in `spur-solver`. It defines a reusable objective-rule contract, makes `rbac.minimum_privilege` executable, and adds `placement.minimize_skew`.

No implementation is authorized until this notebook is reviewed and approved.

## Decision

SPUR will extend Z3 Optimize through **explicit, opt-in objective rules**. Existing hard rules retain their current meaning and continue to define the feasible set.

Wave 1 contains:

1. A catalog and compiler contract for synthesis-only objective rules.
2. `rbac.minimum_privilege`, minimizing caller-declared role-grant cost while required permission reachability and separation-of-duty rules remain hard.
3. `placement.minimize_skew`, minimizing maximum pairwise topology skew while replica conservation and any bound/capacity rules remain hard.

Wave 1 supports one objective binding per `solve_rules` request, using existing lexicographic priority and one collected solution. Multi-objective Pareto/box behavior is deferred.

## Goals and non-goals

### Goals

- Return a proven optimum only when Z3 reports complete optimization termination.
- Keep utility, costs, required permissions, and candidate unknowns caller-owned.
- Preserve existing verify/synthesize outcomes and the raw solver envelope.
- Make objective capability discoverable through versioned rule manifests.
- Require executable conformance vectors and ratcheted bound tests.
- Introduce no behavior change for existing requests.

### Non-goals

- No generic user-authored objective expression inside `solve_rules`.
- No inference that fewer replicas, newer versions, larger targets, or fewer components are preferable.
- No softening of safety, integrity, authorization, capacity, compatibility, or accessibility constraints.
- No multi-objective API, Pareto/box enumeration, unbounded optimization, or cross-family objectives.
- No minimum-repair, configuration-cost, workflow-shortest-path, design-remediation, or further scheduling objectives in Wave 1.

## Current architecture and selected approach

The generic solver request already carries constraints, objectives, objective priority, and solution limits. Family compilers create it through `rules::primitives::request`; that helper initializes no objectives and fixes priority to `lex`. Scheduling demonstrates the extension seam by adding one minimize objective for makespan.

### Alternatives considered

1. **Explicit objective-rule manifests — selected.** Strong catalog authority, stable inputs, family projections, and auditable semantics.
2. **Generic `preferences` on every family request — rejected for Wave 1.** Flexible, but bypasses rule authority and weakens validation.
3. **Optimization parameters on existing hard rules — rejected.** Conflates feasibility with preference and risks changing verification semantics.

Existing manifests default to execution kind `constraint`. Existing requests compile identically. Only a request binding a new objective rule enters Z3 Optimize.

## Reusable objective-rule contract

### Manifest and registry

Add a backward-compatible manifest field:

```yaml
execution_kind: objective # default: constraint
```

An implemented objective manifest declares synthesis semantics, exact caller-owned utility inputs, mode availability, executable vectors, and an anti-pattern that `sat` without complete termination is not an optimum.

Wave 1 rules are synthesis-only. Binding either in `verify` returns `-32602` before solving.

### Compilation

- Compile ordinary rules into named hard predicates first.
- Allow at most one objective binding in a family request.
- Require at least one declared bounded unknown affected by the objective.
- Require finite validated objective coefficients.
- Emit exactly one typed minimize objective.
- Keep `objective_priority = lex` and current single-solution collection.
- Reject cross-family and duplicate objectives before invoking Z3.

A second runtime feasibility solve is not required. Hard predicates define the feasible region; the objective only ranks models inside it. Separate baseline feasibility is a conformance obligation.

In [ ]:
flowchart TD
    SPEC["`@spec OBJECTIVE-ELIGIBILITY
@type Eligibility = enum[eligible, rejected]
@input synthesize: Bool
@input explicit_utility: Bool
@input bounded_unknowns: Bool
@input hard_rules_declared: Bool
@output status: Eligibility
@requires DOMAIN: true`"]
    ELIGIBLE["`@branch ELIGIBLE
@when synthesize = true and explicit_utility = true and bounded_unknowns = true and hard_rules_declared = true
@ensures ELIGIBLE_STATUS: status = eligible`"]
    REJECTED["`@branch REJECTED
@when synthesize = false or explicit_utility = false or bounded_unknowns = false or hard_rules_declared = false
@ensures REJECTED_STATUS: status = rejected`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> ELIGIBLE --> CHECK
    SPEC --> REJECTED --> CHECK

## Result and optimality semantics

The existing optimization envelope remains authoritative:

- `complete`: the returned finite bound is proven optimal for the encoded request.
- `solution_limit`: a partial collected prefix, never a complete frontier or optimum.
- terminal `unknown` after a model: partial feasible evidence, not optimality.
- initial `unknown`, timeout, error, or ended: inconclusive.
- `unsat`: no feasible assignment satisfies the hard predicates; synthesis remains `infeasible`.

The top-level model is insufficient for an optimality claim. Consumers must read `optimization.solutions[0].objectives` and `optimization.termination`.

Unsatisfiable synthesis continues to avoid fabricated verification attribution. Unsat-core diagnosis remains separate because Optimize does not provide the ordinary combined hard-core contract.

In [ ]:
stateDiagram-v2
    [*] --> Ready
    Ready --> Optimizing: start
    Optimizing --> Complete: complete
    Optimizing --> Partial: limit
    Optimizing --> Inconclusive: inconclusive
    note right of Ready
      @spec OPTIMIZATION-LIFECYCLE
      @type Phase = enum[ready, optimizing, complete, partial, inconclusive]
      @type Event = enum[start, complete, limit, inconclusive]
      @input event: Event
      @state-var phase: Phase
      @state-var claim_optimum: Bool
      @requires PRE: phase = ready and claim_optimum = false
      @state Ready
      @invariant CLAIM_ONLY_COMPLETE: claim_optimum = false or phase = complete
    end note
    note right of Optimizing
      @state Optimizing
      @transition START
      @from Ready
      @to Optimizing
      @event event = start
      @guard phase = ready
      @update phase' = optimizing
      @update claim_optimum' = false
    end note
    note right of Complete
      @state Complete
      @transition COMPLETE
      @from Optimizing
      @to Complete
      @event event = complete
      @guard phase = optimizing
      @update phase' = complete
      @update claim_optimum' = true
    end note
    note right of Partial
      @state Partial
      @transition LIMIT
      @from Optimizing
      @to Partial
      @event event = limit
      @guard phase = optimizing
      @update phase' = partial
      @update claim_optimum' = false
    end note
    note right of Inconclusive
      @state Inconclusive
      @transition INCONCLUSIVE
      @from Optimizing
      @to Inconclusive
      @event event = inconclusive
      @guard phase = optimizing
      @update phase' = inconclusive
      @update claim_optimum' = false
      @verify INIT: prove initiate CLAIM_ONLY_COMPLETE
      @verify PRESERVE_START: prove preserve CLAIM_ONLY_COMPLETE on START
      @verify PRESERVE_COMPLETE: prove preserve CLAIM_ONLY_COMPLETE on COMPLETE
      @verify PRESERVE_LIMIT: prove preserve CLAIM_ONLY_COMPLETE on LIMIT
      @verify PRESERVE_INCONCLUSIVE: prove preserve CLAIM_ONLY_COMPLETE on INCONCLUSIVE
    end note

## Rule contract: `rbac.minimum_privilege`

- Family/profile: `policy / nist_rbac`
- Existing rule ID: `rbac.minimum_privilege`
- Availability: `capability_unavailable → implemented`
- Execution kind: `objective`
- Mode: synthesis only
- Subjects: one or more principal IDs

Extend each scoped principal:

```json
{
  "roles": ["existing-role"],
  "required_permissions": ["read", "write"],
  "grant_costs": {"reader": 1, "writer": 2, "admin": 5}
}
```

Costs are positive integers; all `1` expresses minimum grant count. Every candidate `principal_role` unknown needs a cost.

For every scoped required permission, require a matching `rbac.permission_reachable` binding. Hierarchy and separation-of-duty bindings remain hard. Missing reachability coverage is a compile error.

Minimize the sum of costs for assigned candidate role unknowns. Fixed roles remain fixed. Reject missing utility facts, non-positive costs, unknown IDs, no scoped candidate unknown, verify mode, duplicate objectives, or uncovered required permissions.

## Rule contract: `placement.minimize_skew`

- Family/profile: `resource / topology_placement`
- New rule ID: `placement.minimize_skew`
- Execution kind: `objective`
- Mode: synthesis only
- Subjects: exactly one workload ID

The workload declares a known or bounded-unknown replica count, at least two `domain_counts.<domain>` fields, and bounded unknowns for changeable counts. The current resource unknown ABI already supports these paths.

Introduce one bounded internal integer `skew` and assert:

- domain counts are nonnegative;
- their sum equals replicas;
- `skew >= count_i - count_j` for each declared domain pair.

Existing minimum-domain, capacity, quota, and request-limit bindings remain hard when supplied. Minimize `skew`. Return ordinary domain-count projections plus derived `{"node":"<workload>","field":"topology_skew","value":N}`.

`placement.topology_max_skew` remains an optional hard ceiling. Reject fewer than two domains, missing bounds, inconsistent conservation facts, verify mode, or duplicate objectives.

## Conformance, tests, and release gate

Each objective rule requires:

1. A hard-feasible baseline without the objective.
2. A synthesis vector returning `sat + solution`, one finite exact objective bound, and complete termination.
3. A ratcheted strict-better query returning `unsat`.
4. An infeasible vector returning `unsat + infeasible` without fabricated verification attribution.
5. Verify-mode rejection before solving.
6. Missing/invalid utility and unbounded-unknown rejection.
7. Duplicate-objective rejection.
8. Projection tests asserting invariants and objective values, not arbitrary tie-breaking.
9. Manifest/handler bijection and exact catalog coverage.
10. The complete existing `spur-solver` suite.

A near-limit finite performance fixture must complete within its declared budget; timeout remains inconclusive.

In [ ]:
flowchart TD
    SPEC["`@spec OBJECTIVE-RELEASE-GATE
@type Release = enum[ready, blocked]
@input manifest_valid: Bool
@input native_handler_registered: Bool
@input valid_vector_complete: Bool
@input invalid_vector_complete: Bool
@input objective_bound_tested: Bool
@input verify_rejection_tested: Bool
@input regression_suite_green: Bool
@output status: Release
@requires DOMAIN: true`"]
    READY["`@branch READY
@when manifest_valid = true and native_handler_registered = true and valid_vector_complete = true and invalid_vector_complete = true and objective_bound_tested = true and verify_rejection_tested = true and regression_suite_green = true
@ensures READY_STATUS: status = ready`"]
    BLOCKED["`@branch BLOCKED
@when manifest_valid = false or native_handler_registered = false or valid_vector_complete = false or invalid_vector_complete = false or objective_bound_tested = false or verify_rejection_tested = false or regression_suite_green = false
@ensures BLOCKED_STATUS: status = blocked`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> READY --> CHECK
    SPEC --> BLOCKED --> CHECK

## Rollout and task boundaries

1. **Objective contract and conformance harness:** schema defaulting, registry validation, objective detection, guards, and reusable tests.
2. **Policy rule:** utility facts, reachability coverage, cost objective, projection, manifest, conformance.
3. **Resource rule:** derived skew, pairwise bounds, objective, projection, manifest, conformance.
4. **Integration/documentation:** composition tests, catalog counts, MCP snapshots, migration notes, regression verification.

Tasks 2 and 3 may run in parallel only after Task 1. Task 4 depends on both.

### Migration

- Existing JSON stays valid.
- Existing manifests default to `execution_kind = constraint`.
- `rbac.minimum_privilege` becomes executable.
- `placement.minimize_skew` adds one catalog entry.
- No existing rule changes IDs, formula, diagnostics, or mode semantics.

## Risks and mitigations

- **Utility smuggling:** require mandatory caller facts; prohibit inferred costs and permissions.
- **Feasible presented as optimal:** require complete termination and exact bounds.
- **Objective weakens a hard rule:** objectives rank only models satisfying hard predicates.
- **Search explosion:** preserve caps; bound all unknowns; one objective and one solution.
- **Tie-dependent tests:** assert objective values and predicates, not one tied assignment.
- **Notebook proof limitation:** the live `schedule_optimization` NS profile is unavailable and stale relative to `spur-solver`. Formal cells therefore prove eligibility, lifecycle safety, and release partitions only; actual Optimize evidence remains typed-solver and Rust conformance evidence.

## Review checklist

Confirm:

- Wave 1 is one objective per request with fixed lexicographic priority.
- Both objective rules are synthesis-only.
- Minimum privilege requires explicit permissions, positive costs, and hard reachability bindings.
- Minimum skew targets one workload and all declared domains.
- Existing hard rules remain authoritative.
- Multi-objective behavior and remaining candidate families are deferred.

After approval, close epic `bd-27mx` with this notebook and proof hashes, then invoke writing-plans. Do not implement directly from this notebook.